# Lab 1 - NLP Foundations
### Introduction aux LLMs - ESGI Paris | Séance 1
---
**Thème du lab :** critiques de films  
**Durée estimée :** 1h30 - 2h  
**Objectifs :**
- Manipuler du texte avec spaCy (tokenisation, lemmatisation, stop words)
- Construire des représentations vectorielles (BoW, TF-IDF) et les visualiser
- Explorer les embeddings sémantiques avec Word2Vec
- Obtenir des embeddings contextuels avec un modèle Transformer (Hugging Face)

**Prérequis :** Python 3.8+, les installations ci-dessous

## Installation des dépendances

In [4]:
# A executer une seule fois
import subprocess, sys

packages = [
    "spacy", "scikit-learn", "gensim",
    "matplotlib", "transformers", "torch",
    "datasets"
]
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + packages)
subprocess.check_call([sys.executable, "-m", "spacy", "download", "fr_core_news_sm", "-q"])
print("Installation terminee.")

✔ Download and installation successful
You can now load the package via spacy.load('fr_core_news_sm')
Installation terminee.


---
## Corpus

Nous allons travailler sur un petit corpus de critiques de films en français.  
Ce même corpus sera utilisé tout au long du lab.

In [5]:
corpus = [
    "Parasite est un film brillant qui explore les inégalités sociales avec une mise en scene magistrale.",
    "Interstellar nous emmene dans un voyage spatial époustouflant, mele de science et d'emotion.",
    "Le Fabuleux Destin d'Amelie Poulain est un film poétique, plein de charme et d'inventivite.",
    "Inception propose un scenario complexe sur les reves imbriques, visuellement saisissant.",
    "Titanic est un film romantique et tragique, avec des effets speciaux impressionnants pour l'époque.",
    "Mad Max Fury Road est un film d'action frenetique, visuellement brutal et energique.",
    "Her explore avec delicatesse la relation amoureuse entre un homme et une intelligence artificielle.",
    "Roma est un film intime et poétique, tourne en noir et blanc, sur la vie d'une domestique mexicaine.",
    "The Dark Knight est un film de super-heros sombre et psychologique, domine par le Joker.",
    "Portrait de la Jeune Fille en Feu est un film d'époque francais, sobre et puissant, sur le desir et la liberte.",
]

print(f"Corpus chargé : {len(corpus)} critiques")
for i, c in enumerate(corpus):
    print(f"  [{i}] {c[:70]}...")

Corpus chargé : 10 critiques
  [0] Parasite est un film brillant qui explore les inégalités sociales avec...
  [1] Interstellar nous emmene dans un voyage spatial époustouflant, mele de...
  [2] Le Fabuleux Destin d'Amelie Poulain est un film poétique, plein de cha...
  [3] Inception propose un scenario complexe sur les reves imbriques, visuel...
  [4] Titanic est un film romantique et tragique, avec des effets speciaux i...
  [5] Mad Max Fury Road est un film d'action frenetique, visuellement brutal...
  [6] Her explore avec delicatesse la relation amoureuse entre un homme et u...
  [7] Roma est un film intime et poétique, tourne en noir et blanc, sur la v...
  [8] The Dark Knight est un film de super-heros sombre et psychologique, do...
  [9] Portrait de la Jeune Fille en Feu est un film d'époque francais, sobre...


---
## Partie 1 - Pré-traitement avec spaCy

spaCy est une bibliothèque NLP industrielle très utilisée. Elle fournit des pipelines
pré-entraîné pour de nombreuses langues, incluant la tokenisation, le POS tagging,
la lemmatisation et la reconnaissance d'entités.

**Documentation :** https://spacy.io/api

### 1.1 Chargement du modèle

In [ ]:
import spacy

nlp = spacy.load("fr_core_news_sm")
print("Modèle chargé :", nlp.meta["name"], "| Pipeline :", nlp.pipe_names)

### 1.2 Tokenisation

La tokenisation découpe un texte en unites élémentaires (tokens).
En français, les contractions et élisions sont traitées : "l'IA" -> ["l'", "IA"].

In [ ]:
texte_exemple = corpus[0]
doc = nlp(texte_exemple)

print("Texte :", texte_exemple)
print()
print(f"{'Token':<20} {'POS':<10} {'Lemme':<20} {'Stop word'}")
print("-" * 65)
for token in doc:
    print(f"{token.text:<20} {token.pos_:<10} {token.lemma_:<20} {token.is_stop}")

### 1.3 Lemmatisation et suppression des stop words

**Exercice :** Complétez la fonction `preprocess` qui :
1. Applique le pipeline spaCy au texte
2. Garde uniquement les tokens qui ne sont pas des stop words et qui sont alphabétiques
3. Retourne la liste des **lemmes** en minuscules

In [ ]:
# YOUR CODE HERE
raise NotImplementedError

**Vérification :** appliquer `preprocess` sur tout le corpus.

In [ ]:
corpus_preprocessed = [preprocess(text, nlp) for text in corpus]

print("Corpus pré-traité :")
for i, tokens in enumerate(corpus_preprocessed):
    print(f"  [{i}] {tokens}")

### 1.4 Extraction des entités nommées (bonus)

spaCy détecte automatiquement les entités nommées (personnes, lieux, organisations...).

In [ ]:
texte_ner = "Luc Besson a tourne Le Grand Bleu en France. Ce film avec Jean-Marc Barr est sorti en 1988."
doc_ner = nlp(texte_ner)

print("Entités detectees :")
for ent in doc_ner.ents:
    print(f"  '{ent.text}' -> {ent.label_} ({spacy.explain(ent.label_)})")

---
## Partie 2 - Embeddings BoW et TF-IDF avec scikit-learn

Nous allons représenter chaque critique comme un vecteur numérique,
puis visualiser ces vecteurs en 2D avec une ACP (PCA).

### 2.1 Bag of Words

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
import numpy as np

# On travaille sur les textes bruts (le vectorizer fait sa propre tokenisation)
vectorizer_bow = CountVectorizer(max_features=50)
X_bow = vectorizer_bow.fit_transform(corpus).toarray()

print("Forme de la matrice BoW :", X_bow.shape)
print("(nb_documents x taille_vocabulaire)")
print()
print("Extrait du vocabulaire :", vectorizer_bow.get_feature_names_out()[:15])

### 2.2 TF-IDF

**Exercice :** En vous inspirant du code BoW ci-dessus, construisez une matrice TF-IDF
avec `TfidfVectorizer` (mêmes paramètres : `max_features=50`).  
Affichez la forme de la matrice et les 5 mots avec le score TF-IDF le plus élevé
pour la première critique du corpus.

In [ ]:
# YOUR CODE HERE
raise NotImplementedError

### 2.3 Visualisation par ACP (PCA)

La PCA (Analyse en Composantes Principales) réduit la dimensionnalité d'un vecteur
pour pouvoir le tracer en 2D. On va projeter nos critiques et observer si des films
similaires se regroupent.

In [ ]:
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

titres = [
    "Parasite", "Interstellar", "Amelie", "Inception", "Titanic",
    "Mad Max", "Her", "Roma", "Dark Knight", "Portrait JFF"
]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, (X, label, color) in zip(axes, [
    (X_bow,   "Bag of Words", "#333333"),
    (X_tfidf, "TF-IDF",       "#00C897"),
]):
    pca = PCA(n_components=2, random_state=42)
    X_2d = pca.fit_transform(X)

    ax.scatter(X_2d[:, 0], X_2d[:, 1], color=color, s=80, zorder=3)
    for i, titre in enumerate(titres):
        ax.annotate(
            titre,
            (X_2d[i, 0], X_2d[i, 1]),
            textcoords="offset points",
            xytext=(6, 4),
            fontsize=8,
        )
    ax.set_title(label, fontsize=12, fontweight="bold")
    ax.set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)")
    ax.set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)")
    ax.axhline(0, color="#cccccc", linewidth=0.5)
    ax.axvline(0, color="#cccccc", linewidth=0.5)
    ax.set_facecolor("#fafafa")

plt.suptitle("Projection PCA des critiques de films", fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig("pca_embeddings.png", dpi=150, bbox_inches="tight")
plt.show()
print("Figure sauvegardee : pca_embeddings.png")

**Question de réflexion :** Est-ce que des films que vous percevez comme similaires
se retrouvent proches dans l'espace 2D ? Pourquoi les résultats sont-ils différents
entre BoW et TF-IDF ?

---
## Partie 3 - Word2Vec avec gensim

Word2Vec apprend des vecteurs denses pour chaque mot à partir du contexte.
Notre corpus est petit (10 critiques), ce qui limitera la qualité des embeddings.
Pour compenser, nous utiliserons aussi un modèle pré-entraîné en complément.

### 3.1 Entraînement d'un Word2Vec sur notre corpus

In [ ]:
from gensim.models import Word2Vec

# corpus_preprocessed est la liste de listes de tokens definie en Partie 1
model_w2v = Word2Vec(
    sentences=corpus_preprocessed,
    vector_size=50,    # dimension des vecteurs
    window=3,          # taille de la fenêtre de contexte
    min_count=1,       # inclure tous les mots (petit corpus)
    workers=2,
    epochs=100,
    seed=42,
)

print("Modèle entraîné.")
print("Taille du vocabulaire :", len(model_w2v.wv))
print("Exemple - vecteur de 'film' (10 premières dimensions) :")
print(model_w2v.wv["film"][:10])

### 3.2 Mots les plus proches (plus proches voisins)

**Exercice :** Utilisez `model_w2v.wv.most_similar(mot, topn=5)` pour trouver
les 5 mots les plus proches de "film", "voyage" et "noir" dans notre espace vectoriel.  
Commentez les résultats : sont-ils cohérents ? Pourquoi ?

In [ ]:
# YOUR CODE HERE
raise NotImplementedError

### 3.3 Analogies

**Exercice :** Testez l'analogie classique de Word2Vec sur notre corpus.
La méthode est `model_w2v.wv.most_similar(positive=[...], negative=[...], topn=3)`.

Essayez : `"film" + "poétique" - "action"` -> qu'obtenez-vous ?

Note : sur un petit corpus les résultats seront bruités, c'est normal.

In [ ]:
# YOUR CODE HERE
raise NotImplementedError

### 3.4 Visualisation des embeddings Word2Vec

In [ ]:
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt

mots_a_visualiser = [
    "film", "scene", "scenario", "personnage",
    "poétique", "brutal", "romantique", "sombre",
    "voyage", "reve", "amour", "liberte"
]

# Filtrer les mots presents dans le vocabulaire
mots_presents = [m for m in mots_a_visualiser if m in model_w2v.wv]
vecteurs = [model_w2v.wv[m] for m in mots_presents]

pca = PCA(n_components=2, random_state=42)
coords = pca.fit_transform(vecteurs)

fig, ax = plt.subplots(figsize=(8, 6))
ax.scatter(coords[:, 0], coords[:, 1], color="#00C897", s=60, zorder=3)

for i, mot in enumerate(mots_presents):
    ax.annotate(mot, (coords[i, 0], coords[i, 1]),
                textcoords="offset points", xytext=(6, 4), fontsize=9)

ax.set_title("Embeddings Word2Vec - critiques de films", fontsize=12, fontweight="bold")
ax.set_facecolor("#fafafa")
ax.axhline(0, color="#cccccc", linewidth=0.5)
ax.axvline(0, color="#cccccc", linewidth=0.5)
plt.tight_layout()
plt.savefig("word2vec_viz.png", dpi=150, bbox_inches="tight")
plt.show()

---
## Partie 4 - Premier contact avec les Transformers

Les embeddings Word2Vec sont **statiques** : un mot a toujours le même vecteur,
quel que soit son contexte. Les Transformers produisent des embeddings **contextuels** :
le vecteur d'un mot change selon les mots qui l'entourent.

Nous allons utiliser `sentence-transformers`, une bibliothèque qui encapsule
des modèles Transformer pour produire des embeddings de phrases entiere.

In [6]:
# Installation supplémentaire
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "sentence-transformers"])
print("sentence-transformers installé.")

sentence-transformers installé.


### 4.1 Chargement du modèle

In [7]:
from sentence_transformers import SentenceTransformer

# Modèle multilingue leger (dimension 384)
model_st = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")
print("Modèle chargé.")
print("Dimension des embeddings :", model_st.get_sentence_embedding_dimension())

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1513.55it/s, Materializing param=pooler.dense.weight]                               
BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Modèle chargé.
Dimension des embeddings : 384


### 4.1b Explorer le tokenizer

Avant d'encoder des phrases, regardons comment le modèle découpe le texte en tokens.
Le tokenizer transforme une phrase en identifiants numériques (input_ids). Chaque identifiants correspond à un token. 

Utilise le tokenizer pour observer les tokens générés pour les trois phrases suivantes.

In [9]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")


In [14]:
critiques = corpus[:3]
print(critiques) 

for phrase in critiques:
    tokens = # YOUR CODE HERE
    ids    = # YOUR CODE HERE
    print(f"Phrase  : {phrase}")
    print(f"Tokens  : {tokens}")
    print(f"IDs     : {ids}")
    print(f"Nb tokens : {len(tokens)} pour {len(phrase.split())} mots")
    print()



SyntaxError: invalid syntax (4032437195.py, line 5)

### 4.2 Encoder les critiques

**Exercice :** Utilisez `model_st.encode(corpus)` pour obtenir les embeddings
de toutes les critiques. Affichez la forme du tableau résultant.

In [ ]:
# YOUR CODE HERE
raise NotImplementedError

### 4.3 Similarité cosinus entre critiques

La **similarité cosinus** mesure l'angle entre deux vecteurs.
Elle vaut 1 si les vecteurs sont identiques, 0 s'ils sont orthogonaux.

**Exercice :** Calculez la matrice de similarité cosinus entre toutes les critiques
avec `sklearn.metrics.pairwise.cosine_similarity`, puis affichez-la sous forme
de heatmap.

In [ ]:
# YOUR CODE HERE
raise NotImplementedError

### 4.4 Ambiguïté contextuelle - le cas "banque"

Voici l'une des différences fondamentales entre Word2Vec et les Transformers :
un même mot peut avoir des représentations différentes selon son contexte.

In [ ]:
phrases_ambigues = [
    "La batterie de mon téléphone est complètement déchargée.",
    "Le drummer a cassé une baguette pendant la batterie du solo.",
    "Il faut recharger la batterie de la voiture électrique avant le départ.",
    "La batterie de jazz a enflammé la salle lors du concert.",
]

emb_ambigus = model_st.encode(phrases_ambigues)
sim_ambigue = cosine_similarity(emb_ambigus)

print("Similarité entre les phrases avec 'banque' :")

print("Similarité entre les phrases avec 'batterie' :")
labels = ["batterie (électrique) 1", "batterie (instrument) 1",
          "batterie (électrique) 2", "batterie (instrument) 2",
          ]

for i in range(4):
    for j in range(i+1, 4):
        print(f"  '{labels[i]}'")
        print(f"  '{labels[j]}'")
        print(f"  -> similarité = {sim_ambigue[i,j]:.4f}")

**Question :** Les deux phrases sur l'électricité sont-elles plus similaires entre elles
qu'avec les phrases sur l'instrument de musique ? Qu'est-ce que cela dit sur les embeddings contextuels ?

### 4.5 Visualisation finale - comparaison des méthodes

In [ ]:
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# TF-IDF (Partie 2)
pca = PCA(n_components=2, random_state=42)
coords_tfidf = pca.fit_transform(X_tfidf)
ax = axes[0]
ax.scatter(coords_tfidf[:, 0], coords_tfidf[:, 1], color="#333333", s=70, zorder=3)
for i, t in enumerate(titres):
    ax.annotate(t, (coords_tfidf[i, 0], coords_tfidf[i, 1]),
                textcoords="offset points", xytext=(5, 4), fontsize=8)
ax.set_title("TF-IDF", fontsize=12, fontweight="bold")
ax.set_facecolor("#fafafa")

# Transformer
coords_st = PCA(n_components=2, random_state=42).fit_transform(embeddings)
ax = axes[1]
ax.scatter(coords_st[:, 0], coords_st[:, 1], color="#00C897", s=70, zorder=3)
for i, t in enumerate(titres):
    ax.annotate(t, (coords_st[i, 0], coords_st[i, 1]),
                textcoords="offset points", xytext=(5, 4), fontsize=8)
ax.set_title("Transformer (sentence-transformers)", fontsize=12, fontweight="bold")
ax.set_facecolor("#fafafa")

for ax in axes:
    ax.axhline(0, color="#dddddd", linewidth=0.5)
    ax.axvline(0, color="#dddddd", linewidth=0.5)

plt.suptitle("TF-IDF vs Transformer : projection PCA des critiques", fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig("comparison_methods.png", dpi=150, bbox_inches="tight")
plt.show()

---
## Récapitulatif

Remplis le tableau ci-dessous avec oui ou non dans chaque case.

| Méthode | Vectoriel | Sémantique | Contextuel |
|---|---|---|---|
| Bag of Words | ? | ? | ? |
| TF-IDF | ? | ? | ? |
| Word2Vec | ? | ? | ? |
| Transformer | ? | ? | ? |

